In [1]:
import os
import collections
import re

# Local topography

**Note that the data source files are not on GitHub!**

They are in my `~/local` directory and also in the Dropbox folder with the Abegg files.

In [2]:
GH_BASE = os.path.expanduser('~/github')
ORG = 'etcbc'
REPO = 'dss'
# VERSION = '0.1'

# REPO_DIR = f'{GH_BASE}/{ORG}/{REPO}'

LOCAL_BASE = os.path.expanduser('~/Dropbox')
DATA_DIR = f'{LOCAL}/Abegg Data Files'

META_DIR = f'{LOCAL}/sources/meta'

CHAR_TABLE = f'{META_DIR}/chars.txt'
MAN_TABLE = f'{META_DIR}/mans.txt'

# Sources

We have two source files:

* `dss_bib.txt` with biblical material
* `dss_nonbib.txt` with non-biblical material

Throughout this conversion program
we use `True` for the biblical material and `False` for the other material.

In [3]:
SOURCES = dict(
  bib=True,
  nonbib=False,
)

## Tables

We have two tables for mapping certain values in the source data to other values
in a systematic way:

* a manuscript table in order to represent mamuscript codes by their names.
  In most cases the name *is* the code.
* a character table relating unicode characters to their transliterations.

In [4]:
origFromTrans = {}
bookFromCode = {}

def readChars():
  with open(CHAR_TABLE, encoding = 'utf8') as fh:
    for line in fh:
      (orig, trans) = line.rstrip().split('\t')
      origFromTrans[trans] = orig
  print(f'{len(origFromTrans):>4} characters mapped')


def readBooks():
  with open(MAN_TABLE) as fh:
    for line in fh:
      (code, book) = line.rstrip().split('\t')
      bookFromCode[code] = book
  print(f'{len(bookFromCode):>4} mans mapped')
  
readChars()
readBooks()

  38 characters mapped
 265 mans mapped


# Slurp the data

We take in the data and store the lines in a dictionary of lists of lines,
keyed by the boolean "is biblical?"

In [5]:
lines = collections.defaultdict(list)

def getSourceLines():
    counter = 0

    for (src, isBib) in SOURCES.items():
        with open(f'{DATA_DIR}/{REPO}_{src}.txt', encoding='utf8') as fh:
            for line in fh:
                lines[isBib].append(line.strip())
        print(f'{src:<15}: {len(lines[isBib]):>5} lines')
        print(counter)
        counter += 1
    
getSourceLines()

bib            : 243157 lines
0
nonbib         : 378023 lines
1


In [6]:
print(f'{len(lines[False]):>5} lines')

378023 lines


# Parse lines

We read the stored lines and parse them.

If there are errors, we store all occurrences of them in a dict, keyed by:

* kind of error
* the sore point
* the source (`True` or `False` - biblical or not)

And then we get a list of list indices.

Later when we display the errors

* we replace `True` by `bib` and `False` by `nonbib`
* we display line numbers (one higher than list indices)
* we retrieve the corresponding source line

We only show at most `batch` many lines per error and per sore point and per source.

## Line splitting

We have to split lines differently in both sources.

source | separator | # fields
--- | --- | ---
bib | tab | 5
nonbib | space | 4

Several fields are made up of parts themselves.

**bib**

```Gen 1:19	1Q1 f1:1	w	w◊@Pc	32.5```

* `Gen` is book acronym
* `1` is the chapter
* `19` is the verse
* `1Q1` is the manuscript name
* `f1` *what is this called? folio, part, page?* - f = fragment, the number after the f is the column
* `1` *is the line?* - this is the line number
* `w` is the transcription
* `w◊` is the lexeme
* `Pc` is the morphology code
* `32` is the word number
* `5` *what is this? Sub-number, or part of the wordnumber?* - Accordance used a very irregular numbering system, but it essentially boils down to this: if the word number is followed by '.' with another number than the word is a particle of some kind. Sometimes the word number for a particle is the same number as the previous word but with an appended '.' with another number other

**nonbib**

```CD 1:2,3.1 ky k;Iy_2@Pc```

* `CD` is manuscript name
* `1` is column
* `2` is line
* `3` is line (again! are lines nested, is it a sub-line, *how shall I call it?*) - this is actually the word number. if it has a '.' and another number then the word is a particle that is bound to the following word.
* `ky` is transcription (this goes into the plain text)
* `k;ly_2` is lexeme, (*the `_2` is disambiguation?*) - this keys the lexeme to the lexicon entry when there are more than one
* `Pc` is morphology code

In [7]:
def splitLine(isBib, ln, line):
  parts = line.split('\t' if isBib else ' ')
  expFields = 5 if isBib else 4
  nFields = len(parts)
  if nFields != expFields:
    errors['wrong number of fields'][nFields][isBib].append(ln)
    return False
  return parts

In [8]:
errors = collections.defaultdict(
  lambda: collections.defaultdict(
    lambda: collections.defaultdict(list)
  )
)
batch = 78

def showErrors():
  for (kind, soreSources) in sorted(errors.items()):
    print(f'ERROR {kind}:')
    for (sore, srcOccs) in sorted(soreSources.items()):
      print(f'\t{sore}:')
      for (isBib, occs) in sorted(srcOccs.items()):
        nOccs = len(occs)
        srcStr = 'bib' if isBib else 'nonbib'
        print(f'\t\t{srcStr:<6}: {nOccs:>6}x:')
        for occ in occs[0:batch]:
          print(f'\t\t\t{occ + 1:>6} "{lines[isBib][occ]}"')
        if nOccs > batch:
          print('\t\t\t...')

def readData():
  for isBib in lines:
    for (i, line) in enumerate(lines[isBib]):
      if line.startswith('>'):
        continue
      parts = splitLine(isBib, i, line)
      if not parts:
        continue
    
readData()
showErrors()

ERROR wrong number of fields:
	1:
		nonbib:   1144x:
			 15051 "(fl)"
			 15053 "(fy)"
			 15799 "(fl)"
			 15801 "(fy)"
			 15807 "(fl)"
			 15809 "(fy)"
			 16153 "(fl)"
			 16155 "(fy)"
			 16843 "(fl)"
			 16845 "(fy)"
			 16932 "(fl)"
			 16934 "(fy)"
			 17056 "(fl)"
			 17058 "(fy)"
			 17321 "(fl)"
			 17323 "(fy)"
			 26788 "(fl)"
			 26790 "(fy)"
			 27862 "(fl)"
			 27864 "(fy)"
			 28642 "(fl)"
			 28644 "(fy)"
			 38697 "(fl)"
			 38699 "(fy)"
			 38715 "(fl)"
			 38717 "(fy)"
			 39051 "(fl)"
			 39053 "(fy)"
			 39281 "r"
			 39306 "(fl)"
			 39308 "(fy)"
			 39317 "(fl)"
			 39319 "(fy)"
			 50842 "(fl)"
			 50844 "(fy)"
			 51845 "r"
			 52434 "(fl)"
			 52436 "(fy)"
			 57629 "(fl)"
			 57631 "(fy)"
			 57713 "(f0)"
			 57715 "(fy)"
			 57727 "(fl)"
			 57729 "(fy)"
			 57799 "(f0)"
			 57801 "(fy)"
			 57827 "(fl)"
			 57829 "(fy)"
			 57844 "(fl)"
			 57846 "(fy)"
			 57849 "(f0)"
			 57851 "(fy)"
			 57871 "(f0)"

### Errors 1 - 39281 "r"

The characters "r" follows an interlinear line unless the interlinear line ends at the end of the line, then this character immediately precedes the interlinear line.

In [9]:
for line_num in errors['wrong number of fields'][1][False]:
    if 'r' in lines[False][line_num]:
        lines[False][line_num] = lines[False][line_num]+' - - - - -'
        print(line_num)

39280
51844
60238
64988
66135
67586
69034
70655
83163
83174
105507
105612
107921
108639
108662
108682
109294
112587
132292
137065
148644
151505
151707
151799
152005
152297
152838
152930
153291
160630
160659
160676
161413
188152
209559
213210
213341
213478
213567
219834
224175
226422
227154
227183
231974
232206
232603
232685
233133
233162
234509
234571
234950
240938
243160
246031
258486
258589
270009
270453
298961
298978
315083
322150
326578
326827
346333
347707
348413
368167
372844


### Errors 2 - Extra Space Between Words

> 13082 "1QSa 1:8,6.1" Some words in the manuscripts have extra space between them. These appear to be marked with one Ê per extra space that is found in Accordance. However, those extra spaces are not in the lines dict due to whitespace being stripped out I think. So, here we search for lines with a comma as the only lines with a comma that have this error are those lines that have the extra spaces in the original doc.

In [10]:
for line_num in errors['wrong number of fields'][2][False]:
    if ',' in lines[False][line_num]:
        lines[False][line_num] = lines[False][line_num]+' - -'

### Errors 3 - Empty or partial lines with no tag

> Here we find all lines that return error 3 (e.g. :     4486 "CD 13:17,13.1 [") and add the string 'NaN' to the end of the line, thus adding the fourth field required for non-biblical lines and filling it with NaN. All lines the return error 3 are either empty lines or partial lines that cannot receive a tag.

In [11]:
for line_num in errors['wrong number of fields'][3][False]:
    lines[False][line_num] = lines[False][line_num]+' -'
    print(line_num)

1
2
15
28
47
63
84
105
117
129
151
152
153
173
188
242
270
300
319
321
322
337
338
339
355
368
398
415
433
460
482
483
484
506
508
509
539
574
593
608
639
641
642
652
672
693
694
695
706
734
769
787
820
821
822
851
852
875
876
880
892
909
942
995
996
997
1013
1014
1015
1032
1052
1067
1094
1111
1139
1182
1183
1184
1212
1228
1245
1246
1247
1264
1278
1288
1300
1301
1302
1313
1352
1360
1375
1398
1410
1411
1412
1425
1439
1454
1455
1456
1484
1499
1506
1516
1538
1549
1581
1582
1583
1585
1586
1601
1634
1640
1641
1652
1670
1709
1710
1711
1727
1728
1729
1772
1773
1774
1792
1806
1807
1814
1856
1926
1942
1943
1961
1995
2009
2032
2033
2034
2056
2057
2081
2135
2147
2163
2164
2165
2182
2183
2184
2203
2204
2205
2213
2232
2233
2234
2259
2277
2293
2320
2372
2398
2411
2435
2436
2437
2444
2445
2466
2496
2530
2565
2566
2567
2580
2581
2602
2604
2605
2622
2623
2624
2637
2653
2703
2705
2706
2721
2763
2764
2765
2776
2799
2824
2836
2838
2839
2865
2866
2867
2890
2899
2900
2901
2948
2959
2960
2961
2983
2984
2998


### Error 5: two spaces in a row

> There is one occurance of a nonbiblical line having two spaces in a row: 256841 "4Q491 f36:2,4.1 [\\]  \\\@0". Here, I remove the extra space.

In [12]:
for line_num in errors['wrong number of fields'][5][False]:
    lines[False][line_num] = lines[False][line_num][:(lines[False][line_num].index('  '))]+lines[False][line_num][(lines[False][line_num].index('  '))+1:]

### Error 6: two tabs in a row

> There is one occurance of a biblical line having two tabs in a row after the morph tag slot: "208179 "8Q3 f12_16:17	8Q3 f12_16:17	--	\@0		949". Here, I remove the extra tab.

In [13]:
for line_num in errors['wrong number of fields'][6][True]:
    lines[True][line_num] = lines[True][line_num][:(lines[True][line_num].index('\t\t'))]+lines[True][line_num][(lines[True][line_num].index('\t\t'))+1:]

### Lines without Morph Tag step 1

> Some lines do not have a lexeme or morph tag, here we add two blank columns filled with ' -'

In [14]:
for line in lines[False]:
    if not '@' in line and not '%' in line:
        lines[False][lines[False].index(line)] = line + ' -'

In [15]:
for line in lines[True]:
    if not '@' in line and not '%' in line:
        lines[True][lines[True].index(line)] = line + ' - -'

In [ ]:
with open('dss_nonbib_errors_fixed_step_1.txt', 'w', encoding='utf8') as f:
    for line in lines[False]:
        if not (line.startswith('>') or '' in line):
            f.write("%s\n" % line)
        else:
            pass
        
with open('dss_bib_errors_fixed_step_1.txt', 'w', encoding='utf8') as f:
    for line in lines[True]:
        if not (line.startswith('>') or '' in line):
            f.write("%s\n" % line)
        else:
            pass

### Word Numbers step 2

> Remove word numbers, add new column that marks words bound to the next word with 'B'

In [16]:
for line in lines[False]:
    # ignore lines that are placed at the beginning of scroll line
    if line.startswith('>'):
        print(line)
        pass

    else:
        try:
                # set pattern to match any number of characters before a comma (scroll name + frag num + line num)
            p = re.compile('.*?\,\d+')
            
                # match the pattern to the line and store in variable s1_s
            s1_s = p.match(line).group(0)
            
            try:
                    # try to match the pattern to line + 1 and store in s2_s
                s2_s = p.match(lines[False][lines[False].index(line)+1]).group(0)
            except:
                    # if the pattern is not matched (e.g. line + 1 is the beginning of a new scroll line)
                s2_s = ''
            
                # compare s1_2 and s2_s. 

            if s1_s == s2_s:
                    # if they are the same, remove the word number
                line_update1 = re.sub('\,\d+\.\d+', '', line)
                    # and add ' B'
                lines[False][lines[False].index(line)] = line_update1 + ' B'

            else:
                    # if they are different, remove the word number
                line_update3 = re.sub('\,\d+\.\d+', '', line)
                    # and add ' -'
                lines[False][lines[False].index(line)] = line_update3 + ' -'
        except (AttributeError, IndexError) as e:
            pass

>CD 1:1 -
>CD 1:2 -
>CD 1:3 -
>CD 1:4 -
>CD 1:5 -
>CD 1:6 -
>CD 1:7 -
>CD 1:8 -
>CD 1:9 -
>CD 1:10 -
>CD 1:11 -
>CD 1:12 -
>CD 1:13 -
>CD 1:14 -
>CD 1:15 -
>CD 1:16 -
>CD 1:17 -
>CD 1:18 -
>CD 1:19 -
>CD 1:20 -
>CD 1:21 -
>CD 2:1 -
>CD 2:2 -
>CD 2:3 -
>CD 2:4 -
>CD 2:5 -
>CD 2:6 -
>CD 2:7 -
>CD 2:8 -
>CD 2:9 -
>CD 2:10 -
>CD 2:11 -
>CD 2:12 -
>CD 2:13 -
>CD 2:14 -
>CD 2:15 -
>CD 2:16 -
>CD 2:17 -
>CD 2:18 -
>CD 2:19 -
>CD 2:20 -
>CD 2:21 -
>CD 3:1 -
>CD 3:2 -
>CD 3:3 -
>CD 3:4 -
>CD 3:5 -
>CD 3:6 -
>CD 3:7 -
>CD 3:8 -
>CD 3:9 -
>CD 3:10 -
>CD 3:11 -
>CD 3:12 -
>CD 3:13 -
>CD 3:14 -
>CD 3:15 -
>CD 3:16 -
>CD 3:17 -
>CD 3:18 -
>CD 3:19 -
>CD 3:20 -
>CD 3:21 -
>CD 4:1 -
>CD 4:2 -
>CD 4:3 -
>CD 4:4 -
>CD 4:5 -
>CD 4:6 -
>CD 4:7 -
>CD 4:8 -
>CD 4:9 -
>CD 4:10 -
>CD 4:11 -
>CD 4:12 -
>CD 4:13 -
>CD 4:14 -
>CD 4:15 -
>CD 4:16 -
>CD 4:17 -
>CD 4:18 -
>CD 4:19 -
>CD 4:20 -
>CD 4:21 -
>CD 5:1 -
>CD 5:2 -
>CD 5:3 -
>CD 5:4 -
>CD 5:5 -
>CD 5:6 -
>CD 5:7 -
>CD 5:8 -
>CD 5:9 -
>CD 5:10 -
>CD 5:11 -


In [17]:
for line in lines[True]:
    p = re.compile('\t\d+\.?\d?')
    word_number = p.findall(line)[-1]

    if '.' in word_number:
        lines[True][lines[True].index(line)] = re.sub('\d+\.\d', 'B', line)
    else:
        lines[True][lines[True].index(line)] = re.sub(word_number, '\t-', line)

In [ ]:
with open('dss_nonbib_errors_fixed_step_2.txt', 'w', encoding='utf8') as f:
    for line in lines[False]:
        if not (line.startswith('>') or '' in line):
            f.write("%s\n" % line)
        else:
            pass
        
with open('dss_bib_errors_fixed_step_2.txt', 'w', encoding='utf8') as f:
    for line in lines[True]:
        if not (line.startswith('>') or '' in line):
            f.write("%s\n" % line)
        else:
            pass

### Errors 2 - Interlinear lines step 3

> Interlinear lines or words are marked with (a) or in two cases where two lines are interlinear a (b) for the second oen. The line/word will be preceded by "e(s9).(xb5). (a)" and the end of the line/word will be followed by "r"
> interate through each line. Create flag - interlinear = false. Triggered to true if line contains "e(s9).(xb5). (a)" or "e(s9).(xb5). (b)" triggered to false at escr and end of line. Add '-' if false, add ' s1' or ' s2' if true.

In [18]:
is_s = '-'

for line in lines[False]:
    if line.startswith('>'):
        is_s = '-'

    elif '. (a)' in line:
        is_s = 's1'

    elif '. (b)' in line:
        is_s = 's2'

    elif 'r' in line:
        is_s = '-'

    else:
        if is_s == '-':
            lines[False][lines[False].index(line)] = line + ' -'
        elif is_s == 's1':
            lines[False][lines[False].index(line)] = line + ' s1'
        else:
            lines[False][lines[False].index(line)] = line + ' s2'


In [ ]:
with open('dss_nonbib_errors_fixed_step_3.txt', 'w', encoding='utf8') as f:
    for line in lines[False]:
        if not (line.startswith('>') or '' in line):
            f.write("%s\n" % line)
        else:
            pass
        
with open('dss_bib_errors_fixed_step_3.txt', 'w', encoding='utf8') as f:
    for line in lines[True]:
        if not (line.startswith('>') or '' in line):
            f.write("%s\n" % line)
        else:
            pass

 ### Errors 1 - Different Fonts in non-biblical texts step 4

> Error mostly contains lines that are wrapped around words that appear in the Accordance data in a font other than Yehudit. The most common is Lachish '(Fl)' which is used for words that are written in Paleo-Hebrew. Here we replace the wrapper lines with the string 'remove 2 3 4' to tag it for removal at the end of the error cleansing process. The line needs to remain until the end of the process so that the indexing of lines remains keyed to the indexing of the errors. Then, we add the string 'PH' (Paleo-Hebrew) to the end of the Paleo-Hebrew line as an additional tag for words wrapped in '(Pl)'. For words wrapped in '(F0)' we append 'F0' to the end of the tagged as a place holder until I figure out what font (F0) stands for. (FT) only occurs in four lines, not sure what it stands for. But I think it sets the font for the entire line. Here, I add (FT) to the tag of the first word with a tag and delete the line setting the font (note that this error occurs in Errors 2).

In [19]:
font = '-'

for line in lines[False]:
    if line.startswith('>'):
        font = '-'

    elif '(fl)' in line:
        font = 'PH'

    elif '(fy)' in line:
        font = '-'

    elif '(f0)' in line:
        font = 'f0'
        
    elif '(ft)' in line:
        font = 'FT'

    else:
        if font == 'PH':
            lines[False][lines[False].index(line)] = line + ' PH'
        elif font == 'f0':
            lines[False][lines[False].index(line)] = line + ' F0'
        elif font == 'ft':
            lines[False][lines[False].index(line)] = line + ' FT'            
        else:
            lines[False][lines[False].index(line)] = line + ' -'

In [ ]:
with open('dss_nonbib_errors_fixed_step_4.txt', 'w', encoding='utf8') as f:
    for line in lines[False]:
        if not (line.startswith('>') or '' in line):
            f.write("%s\n" % line)
        else:
            pass
        
with open('dss_bib_errors_fixed_step_4.txt', 'w', encoding='utf8') as f:
    for line in lines[True]:
        if not (line.startswith('>') or '' in line):
            f.write("%s\n" % line)
        else:
            pass

### Errors 3 - Biblical lines in different fonts step 5

> Some biblical lines are in Paleo-Hebrew. Here we delete the wrapper lines and add 'PH' to the end of the morph tag for the word that is in Paleo-Hebrew.

In [20]:
font = '-'

for line in lines[True]:
    if line.startswith('>'):
        font = '-'

    elif '(fl)' in line:
        font = 'PH'

    elif '(fy)' in line:
        font = '-'

    else:
        if font == 'PH':
            lines[True][lines[True].index(line)] = line + '\tPH'           
        else:
            lines[True][lines[True].index(line)] = line + '\t-'

In [ ]:
with open('dss_nonbib_errors_fixed_step_5.txt', 'w', encoding='utf8') as f:
    for line in lines[False]:
        if not (line.startswith('>') or '' in line):
            f.write("%s\n" % line)
        else:
            pass
        
with open('dss_bib_errors_fixed_step_5.txt', 'w', encoding='utf8') as f:
    for line in lines[True]:
        if not (line.startswith('>') or '' in line):
            f.write("%s\n" % line)
        else:
            pass

### Add Column for Hebrew/Aramaic step 6

In [21]:
for line in lines[False]:
    if not line.startwith('>'):
        if '%' in line:
            lines[False][lines[False].index(line)] = line + ' A'

        elif '@' in line:
            lines[False][lines[False].index(line)] = line + ' H'
            
        else:
            lines[False][lines[False].index(line)] = line + '\t-'

In [22]:
for line in lines[True]:
    if not line.startwith('>'):
        if '%' in line:
            lines[True][lines[True].index(line)] = line + ' A'

        elif '@' in line:
            lines[True][lines[True].index(line)] = line + ' H'
            
        else:
            lines[True][lines[True].index(line)] = line + '\t-'

In [ ]:
with open('dss_nonbib_errors_fixed_step_6.txt', 'w', encoding='utf8') as f:
    for line in lines[False]:
        if not (line.startswith('>') or '' in line):
            f.write("%s\n" % line)
        else:
            pass
        
with open('dss_bib_errors_fixed_step_6.txt', 'w', encoding='utf8') as f:
    for line in lines[True]:
        if not (line.startswith('>') or '' in line):
            f.write("%s\n" % line)
        else:
            pass

### Adjust and Create columns step 7

> Bib columns (11):
1. Book 
2. chapter
3. verse
4. Scroll
5. fragment
6. line
7. Transcription
8. Lexeme
9. morph tag
10. Bound (particles that are bound to the next word are marked with a 'B')
11. Script (PH= Paleohebrew)
12. Language (H = Hebrew, A = Aramaic)

>Non bib columns (9):
1. Scroll Name
2. Column
3. line
4. Transcription
5. Lexeme
6. morph tag
7. Bound (particles that are bound to the next word are marked with a 'B')
8. Interlinear (s1 and s2)
9. Script (PH = Paleohebrew, F0 and FT = not sure what these stand for, but kept them so more research can be done).
10. Language (H = Hebrew, A = Aramaic)


In [23]:
for line in lines[False]:
    updated_line1 = re.sub(' ', '\t', line)
    updated_line2 = re.sub('\:', '\t', updated_line1)
    updated_line3 = re.sub('\@', '\t', updated_line2)
    updated_line4 = re.sub('\%', '\t', updated_line3)

    
    lines[False][lines[False].index(line)] = updated_line4

In [24]:
for line in lines[True]:
    updated_line1 = re.sub('\t ', '\t', line)
    updated_line2 = re.sub(' \t ', '\t', updated_line1)
    updated_line3 = re.sub('\t\t', '\t', updated_line2)
    updated_line4 = re.sub('  ', '\t', updated_line3)
    updated_line5 = re.sub(' ', '\t', updated_line4)
    updated_line6 = re.sub('\:', '\t', updated_line5)
    updated_line7 = re.sub('\%\@', '\t', updated_line6)
    updated_line8 = re.sub('\@', '\t', updated_line7, 1)
    updated_line9 = re.sub('\%', '\t', updated_line8, 1)
    
    lines[True][lines[True].index(line)] = updated_line9

In [ ]:
with open('dss_nonbib_errors_fixed_step_7.txt', 'w', encoding='utf8') as f:
    for line in lines[False]:
        if not (line.startswith('>') or '' in line):
            f.write("%s\n" % line)
        else:
            pass
        
with open('dss_bib_errors_fixed_step_7.txt', 'w', encoding='utf8') as f:
    for line in lines[True]:
        if not (line.startswith('>') or '' in line):
            f.write("%s\n" % line)
        else:
            pass

### Fix specific errors

In [25]:
lines[True][lines[True].index('Is\t7\t22\t1Q8\t3\t3\tjm|\xa0\xa0a|h|\t\tjRmVaDh\tncfs\t-\t-')] = 'Is\t7\t22\t1Q8\t3\t3\tjm|\xa0\xa0a|h|\tjRmVaDh\tncfs\t-\t-'

lines[True][lines[True].index('Is	64	4	1Q8	27	30	owlM[	oøwlDMXncms	-	-	-	-')] = 'Is	64	4	1Q8	27	30	owlM[	oøwlDM\tncms	-	-'

lines[True][lines[True].index('Deut	20	7	4Q33	f37	1	ayC		aIyv	ncms	-	-')] = 'Deut	20	7	4Q33	f37	1	ayC	aIyv	ncms	-	-'

lines[True][lines[True].index('2Sam	3	26	4Q51	f61i+62	6	sy[rh	sIr∂h		np	-	-')] = '2Sam	3	26	4Q51	f61i+62	6	sy[rh	sIr∂h	np	-	-'

lines[True][lines[True].index('Judg	1	11	XJudges	f1	4	q]ryt		qIr√yÅt	np	-	-')] = 'Judg	1	11	XJudges	f1	4	q]ryt	qIr√yÅt	np	-	-'

ValueError: 'Is\t7\t22\t1Q8\t3\t3\tjm|\xa0\xa0a|h|\t\tjRmVaDh\tncfs\t-\t-' is not in list

In [ ]:
lines[True][lines[True].index('Deut	11	10	1Q4	f28	1	w	w◊	B	-	-	-')] = 'Deut	11	10	1Q4	f28	1	w	w◊	B	-	-'

lines[True][lines[True].index('Ps	44	8	1Q12	f1_4	6	w	w◊	B	-	-	-')] = 'Ps	44	8	1Q12	f1_4	6	w	w◊	B	-	-'

### Create new File with errors fixed

In [26]:
with open('dss_nonbib_errors_fixed.txt', 'w', encoding='utf8') as f:
    for line in lines[False]:
        if not (line.startswith('>') or '' in line):
            f.write("%s\n" % line)
        else:
            pass
        
with open('dss_bib_errors_fixed.txt', 'w', encoding='utf8') as f:
    for line in lines[True]:
        if not (line.startswith('>') or '' in line):
            f.write("%s\n" % line)
        else:
            pass

In [27]:
for line in lines[False]:
    if not (line.startswith('>') or '' in line):
        if line.count('\t') != 9:
            print(line)

CD	1	1	≥	-	-	-	-	-
CD	1	1	≤	-	-	-	-	-
CD	1	2	.	-	-	-	-	-
CD	1	2	.	-	-	-	-	-
CD	1	4	.	-	-	-	-	-
CD	1	5	.	-	-	-	-	-
CD	1	7	.	-	-	-	-	-
CD	1	8	.	-	-	-	-	-
CD	1	9	.	-	-	-	-	-
CD	1	10	.	-	-	-	-	-
CD	1	11	.	-	-	-	-	-
CD	1	11	≥	-	-	-	-	-
CD	1	11	≤	-	-	-	-	-
CD	1	13	.	-	-	-	-	-
CD	1	14	.	-	-	-	-	-
CD	1	18	.	-	-	-	-	-
CD	1	19	.	-	-	-	-	-
CD	1	21	.	-	-	-	-	-
CD	2	1	.	-	-	-	-	-
CD	2	2	≥	-	-	-	-	-
CD	2	2	≤	-	-	-	-	-
CD	2	3	.	-	-	-	-	-
CD	2	3	≥	-	-	-	-	-
CD	2	3	≤	-	-	-	-	-
CD	2	4	.	-	-	-	-	-
CD	2	5	.	-	-	-	-	-
CD	2	7	.	-	-	-	-	-
CD	2	8	.	-	-	-	-	-
CD	2	9	.	-	-	-	-	-
CD	2	10	.	-	-	-	-	-
CD	2	12	.	-	-	-	-	-
CD	2	12	≥	-	-	-	-	-
CD	2	12	≤	-	-	-	-	-
CD	2	13	.	-	-	-	-	-
CD	2	14	≥	-	-	-	-	-
CD	2	14	≤	-	-	-	-	-
CD	2	15	.	-	-	-	-	-
CD	2	17	.	-	-	-	-	-
CD	2	18	.	-	-	-	-	-
CD	2	19	.	-	-	-	-	-
CD	2	21	.	-	-	-	-	-
CD	3	1	≥	-	-	-	-	-
CD	3	1	≤	-	-	-	-	-
CD	3	1	.	-	-	-	-	-
CD	3	3	.	-	-	-	-	-
CD	3	4	.	-	-	-	-	-
CD	3	4	≥	-	-	-	-	-
CD	3	4	≤	-	-	-	-	-
CD	3	5	.	-	-	-	-	-
CD	3	6	.	-	-	-	-	-
CD	3	9	.	-	-	-	-	-
CD	3	10	.	

In [28]:
for line in lines[True]:
    if not (line.startswith('>') or '' in line):
        if line.count('\t') != 11:
            print(line)

Gen	1	18	1Q1	f1	1	.	-	-	-	-
Gen	1	19	1Q1	f1	1	.	-	-	-	-
Gen	1	20	1Q1	f1	1	/	-	-	-	-
Gen	1	20	1Q1	f1	2	.	-	-	-	-
Gen	1	20	1Q1	f1	2	[	-	-	-	-
Gen	1	20	1Q1	f1	2	/	-	-	-	-
Gen	1	21	1Q1	f1	3	.	-	-	-	-
Gen	1	21	1Q1	f1	3	/	-	-	-	-
Gen	3	11	1Q1	f2	1	/	-	-	-	-
Gen	3	11	1Q1	f2	2	.	-	-	-	-
Gen	3	12	1Q1	f2	2	.	-	-	-	-
Gen	3	13	1Q1	f2	2	/	-	-	-	-
Gen	3	13	1Q1	f2	3	.	-	-	-	-
Gen	3	14	1Q1	f2	3	/	-	-	-	-
Gen	3	14	1Q1	f2	4	/	-	-	-	-
Gen	3	14	1Q1	f2	5	.	-	-	-	-
Gen	3	14	1Q1	f2	5	[	-	-	-	-
Gen	3	14	1Q1	f2	5	/	-	-	-	-
Gen	22	13	1Q1	f3	1	.	-	-	-	-
Gen	22	13	1Q1	f3	1	[	-	-	-	-
Gen	22	13	1Q1	f3	1	/	-	-	-	-
Gen	22	14	1Q1	f3	2	/	-	-	-	-
Gen	22	14	1Q1	f3	3	.	-	-	-	-
Gen	22	15	1Q1	f3	3	.	-	-	-	-
Gen	22	15	1Q1	f3	3	[	-	-	-	-
Gen	22	15	1Q1	f3	3	/	-	-	-	-
Gen	23	17	1Q1	f4	1	]	-	-	-	-
Gen	23	17	1Q1	f4	1	/	-	-	-	-
Gen	23	17	1Q1	f4	2	/	-	-	-	-
Gen	23	17	1Q1	f4	3	.	-	-	-	-
Gen	23	18	1Q1	f4	3	/	-	-	-	-
Gen	23	18	1Q1	f4	4	.	-	-	-	-
Gen	23	19	1Q1	f4	4	/	-	-	-	-
Gen	23	19	1Q1	f4	5	.	-	-	-	-
Gen	23	19	1Q1	f4	5	[	-	-	-	-
Gen

### tests

In [ ]:
lines[True][lines[True].index(Is	7	22	1Q8	3	3	jm|  a|h|	jRmVaDh	ncfs	-	-)] = line + '\tPH'

In [ ]:
lines[True].index('Is 7:22	1Q8 3:3	jm|  a|h| 	jRmVaDh@ncfs	31')

In [ ]:
lines[True][32131]

In [ ]:
# non biblical - fixed

1QSa 1:8,6.1 ÊÊ

# biblical errors

# four extra columns intead of three, original has no morph tag - fixed
Deut	11	10	1Q4	f28	1	w	w◊	B	-	-	-
Deut 11:10	1Q4 f28:1	w	w◊	14729.7

# double space after vertical line and two tabs - fixed
Is	7	22	1Q8	3	3	jm|  a|h|		jRmVaDh	ncfs	-	-
Is 7:22	1Q8 3:3	jm|  a|h| 	jRmVaDh@ncfs	31

# four extra columns instead of three - X should be @ - fixed
Is	64	4	1Q8	27	30	owlM[	oøwlDMXncms	-	-	-	-
Is 64:4	1Q8 27:30	owlM[	oøwlDMXncms	25242

# four extra columns intead of three, original has no morph tag - fixed
Ps	44	8	1Q12	f1_4	6	w	w◊	B	-	-	-
Ps 44:8	1Q12 f1_4:6	w	w◊	401.5

# two tabs in a row, original has space + tab - fixed
Deut	20	7	4Q33	f37	1	ayC		aIyv	ncms	-	-
Deut 20:7	4Q33 f37:1	ayC 	aIyv@ncms	15429

# two tabs in a row, original has two spaces -fixed
2Sam	3	26	4Q51	f61i+62	6	sy[rh	sIr∂h		np	-	-
2Sam 3:26	4Q51 f61i+62:6	sy[rh	sIr∂h  @np	5764


# two tabs, original has space + tab
Judg	1	11	XJudges	f1	4	q]ryt		qIr√yÅt	np	-	-
Judg 1:11	XJudges f1:4	q]ryt 	qIr√yÅt@np	1221


In [ ]:
string = 'CD	1	1	w	w◊	Pc	B	-	-'

string.count('\t')

In [ ]:
Eccl	1	15	4Q110	f1_3	10	l	lV	Pp	0	B	-
Eccl 1:15	4Q110 f1_3:10	l	lV@Pp@0	409.5
        
Is	7	22	1Q8	3	3	jm|  a|h|		jRmVaDh	ncfs	-	-
Is 7:22	1Q8 3:3	jm|  a|h| 	jRmVaDh@ncfs	31




In [ ]:
list = ['>4Q559 f12:1',
'4Q559 f12:1 ] - - - -',
'4Q559 f12:1 -- \%0 B - -',
'(fl) -',
'4Q559 f12:1 [D|] 02%uc000 - - PH',
'(fy) -',
'4Q559 f12:1 -- \%0 - - -']

for line in list:
    if not ('\@' in line or '%' in line):
    # if not ('4' in line or 'fl' in line):
        list[list.index(line)] = line + ' T'
print(list)

In [ ]:
string = 'CD	5	3	}}np\\{{	p\\@vnPmsa	-	-	-'

s_update = re.sub('\@', '\t', string)

s_update

In [ ]:
Judg 1:11	XJudges f1:4	q]ryt 	qIr√yÅt@np	1221
